# Phase 2c: Selección quirúrgica + Regularización fuerte

**Lección de Fase 2b**: más features → OOF sube pero LB baja (overfitting a los folds).

**Estrategia**: partir de Fase 2 (mejor LB: 0.82174) y agregar solo:
- `n_tests_completed` — feature nueva conceptualmente distinta (cuántos tests realizó, calculada pre-imputación)
- 5 interactions con mayor ganancia incremental y menor redundancia: `z_sprint_x_bench`, `speed_agility_ratio`, `strength_per_weight`, `agility_shuttle_ratio`, `power_speed`
- Regularización más fuerte: `min_child_samples=40`, `num_leaves=47` (alineado con lo que Optuna encontró)

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import warnings
from pathlib import Path
from datetime import datetime
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')

SEED     = 42
N_FOLDS  = 5
TE_ALPHA = 10

INPUT_PATH  = Path('/Users/haroldlagares/Downloads/competition/input')
RESULTS_DIR = Path('/Users/haroldlagares/Downloads/competition/results')
RESULTS_DIR.mkdir(exist_ok=True)
print('LightGBM:', lgb.__version__)

## 2. Load Data

In [ ]:
train_raw = pd.read_csv(INPUT_PATH / 'train.csv')
test_raw  = pd.read_csv(INPUT_PATH / 'test.csv')
print('Train:', train_raw.shape, '| Test:', test_raw.shape)

## 3. Preprocessing

In [ ]:
NULL_COLS    = ['Age', 'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle']
PERF_COLS    = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle']
INVERSE_COLS = ['Sprint_40yd', 'Agility_3cone', 'Shuttle']
TE_COLS      = ['School', 'Position', 'Position_Type']

train = train_raw.copy()
test  = test_raw.copy()

# --- n_tests_completed: ANTES de imputar ---
for df in [train, test]:
    df['n_tests_completed'] = df[PERF_COLS].notna().sum(axis=1)

print('Draft rate por n_tests_completed:')
print(train.groupby('n_tests_completed')['Drafted'].agg(['mean','count']).round(3))

# --- Missing flags ---
for col in NULL_COLS:
    train[f'missing_{col}'] = train[col].isna().astype(np.int8)
    test[f'missing_{col}']  = test[col].isna().astype(np.int8)

# --- Imputación por grupo ---
group_medians   = train.groupby('Position_Type')[NULL_COLS].median()
global_fallback = group_medians.median()
for col in NULL_COLS:
    for df in [train, test]:
        mask = df[col].isna()
        if mask.any():
            df.loc[mask, col] = df.loc[mask, 'Position_Type'].map(group_medians[col]).fillna(global_fallback[col])

# --- BMI ---
for df in [train, test]:
    df['BMI'] = df['Weight'] / (df['Height'] ** 2)

# --- Z-scores por Position ---
pos_stats = {}
for col in PERF_COLS:
    pos_stats[col] = {
        'mean': train.groupby('Position')[col].mean(),
        'std':  train.groupby('Position')[col].std().replace(0, np.nan).fillna(1),
    }
for df in [train, test]:
    for col in PERF_COLS:
        z = (df[col] - df['Position'].map(pos_stats[col]['mean'])) / df['Position'].map(pos_stats[col]['std']).replace(0, np.nan).fillna(1)
        df[f'z_{col}'] = -z if col in INVERSE_COLS else z

z_cols = [f'z_{c}' for c in PERF_COLS]
for df in [train, test]:
    df['overall_athleticism'] = df[z_cols].mean(axis=1)

# --- Label encode Player_Type ---
le = LabelEncoder()
le.fit(pd.concat([train['Player_Type'], test['Player_Type']], ignore_index=True).astype(str))
train['Player_Type'] = le.transform(train['Player_Type'].astype(str))
test['Player_Type']  = le.transform(test['Player_Type'].astype(str))

# --- Target encoding estático ---
def target_encode_static(train_df, apply_df, col, target='Drafted', alpha=10):
    global_mean = train_df[target].mean()
    stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    stats['te'] = (stats['mean'] * stats['count'] + global_mean * alpha) / (stats['count'] + alpha)
    return apply_df[col].map(stats['te']).fillna(global_mean)

for col in TE_COLS:
    train[f'te_{col}'] = target_encode_static(train_raw, train, col, 'Drafted', TE_ALPHA)
    test[f'te_{col}']  = target_encode_static(train_raw, test,  col, 'Drafted', TE_ALPHA)

# --- 5 interactions seleccionadas (post-imputación) ---
for df in [train, test]:
    df['z_sprint_x_bench']    = df['z_Sprint_40yd'] * df['z_Bench_Press_Reps']
    df['speed_agility_ratio'] = df['Sprint_40yd'] / df['Agility_3cone']
    df['strength_per_weight'] = df['Bench_Press_Reps'] / df['Weight']
    df['agility_shuttle_ratio'] = df['Agility_3cone'] / df['Shuttle']
    df['power_speed']         = df['Weight'] / df['Sprint_40yd']

print('\nPreprocessing completo. Nulos en NULL_COLS:', train[NULL_COLS].isnull().sum().sum())

## 4. Feature Set

In [ ]:
FEATURE_COLS = [
    # Base Fase 2 (29)
    'Year', 'Age', 'Height', 'Weight',
    'Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle',
    'Player_Type',
    'missing_Age', 'missing_Sprint_40yd', 'missing_Vertical_Jump', 'missing_Bench_Press_Reps',
    'missing_Broad_Jump', 'missing_Agility_3cone', 'missing_Shuttle',
    'BMI',
    'z_Sprint_40yd', 'z_Vertical_Jump', 'z_Bench_Press_Reps', 'z_Broad_Jump', 'z_Agility_3cone', 'z_Shuttle',
    'overall_athleticism',
    'te_School', 'te_Position', 'te_Position_Type',
    # Nuevas (6)
    'n_tests_completed',
    'z_sprint_x_bench',
    'speed_agility_ratio',
    'strength_per_weight',
    'agility_shuttle_ratio',
    'power_speed',
]

TARGET_COL = 'Drafted'
y      = train[TARGET_COL].values
X      = train[FEATURE_COLS].values
X_test = test[FEATURE_COLS].values

print(f'Total features: {len(FEATURE_COLS)} (Fase 2: 29, nuevas: 6)')

## 5. LightGBM — Regularización más fuerte

Alineado con lo que Optuna encontró en Fase 3: `min_child_samples` grande y `num_leaves` conservador previenen overfitting en ~2800 filas.

In [ ]:
lgb_params = {
    'objective':         'binary',
    'metric':            'auc',
    'learning_rate':     0.05,
    'num_leaves':        47,       # Fase 2: 63 → reducido
    'max_depth':         -1,
    'min_child_samples': 40,       # Fase 2: 20 → aumentado (Optuna encontró 75)
    'feature_fraction':  0.8,
    'bagging_fraction':  0.8,
    'bagging_freq':      5,
    'reg_alpha':         0.1,
    'reg_lambda':        0.1,
    'verbose':           -1,
    'seed':              SEED,
}

skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
fold_scores = []
feature_importances = pd.DataFrame({'feature': FEATURE_COLS})

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    dtrain = lgb.Dataset(X[train_idx], label=y[train_idx], feature_name=FEATURE_COLS)
    dval   = lgb.Dataset(X[val_idx],   label=y[val_idx],   feature_name=FEATURE_COLS, reference=dtrain)

    model = lgb.train(
        lgb_params, dtrain,
        num_boost_round=1000,
        valid_sets=[dval],
        callbacks=[
            lgb.early_stopping(50, verbose=False),
            lgb.log_evaluation(-1),
        ],
    )

    val_pred = model.predict(X[val_idx])
    fold_auc = roc_auc_score(y[val_idx], val_pred)
    fold_scores.append(fold_auc)

    oof_preds[val_idx] = val_pred
    test_preds += model.predict(X_test) / N_FOLDS
    feature_importances[f'fold_{fold}'] = model.feature_importance(importance_type='gain')

    print(f'Fold {fold} | AUC: {fold_auc:.5f} | Best iter: {model.best_iteration}')

oof_auc = roc_auc_score(y, oof_preds)
print(f'\nOOF AUC: {oof_auc:.5f}  (±{np.std(fold_scores):.5f})')
print(f'OOF-LB gap objetivo: < 0.003 (Fase 2 tuvo 0.003, Fase 2b tuvo 0.019)')

## 6. Feature Importance

In [ ]:
NEW_FEATURES = ['n_tests_completed','z_sprint_x_bench','speed_agility_ratio',
                'strength_per_weight','agility_shuttle_ratio','power_speed']

fold_cols = [c for c in feature_importances.columns if c.startswith('fold_')]
feature_importances['mean_gain'] = feature_importances[fold_cols].mean(axis=1)
feature_importances = feature_importances.sort_values('mean_gain', ascending=False)
feature_importances['is_new'] = feature_importances['feature'].isin(NEW_FEATURES)

plt.figure(figsize=(11, 8))
colors = ['#e74c3c' if n else '#3498db' for n in feature_importances['is_new']]
plt.barh(feature_importances['feature'], feature_importances['mean_gain'], color=colors)
from matplotlib.patches import Patch
plt.legend(handles=[Patch(color='#e74c3c', label='Nueva'), Patch(color='#3498db', label='Base Fase 2')])
plt.xlabel('Mean Gain')
plt.title('Feature Importance — Phase 2c')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print(feature_importances[['feature','mean_gain','is_new']].to_string(index=False))

## 7. Guardar Submission

In [ ]:
timestamp = datetime.now().strftime('%Y-%m-%d_%H-%M-%S')
submission_path = RESULTS_DIR / f'submission_phase2c_{timestamp}.csv'

submission = pd.read_csv(INPUT_PATH / 'sample_submission.csv')
submission['Drafted'] = test_preds
submission.to_csv(submission_path, index=False)

print(f'Submission: {submission_path}')
print(f'OOF AUC: {oof_auc:.5f}')
print(submission.head())